# ⚡ Business Entity Resolution — Fast & Memory-Safe Kaggle Runbook

This runbook is configured to be **100% memory-safe** on Kaggle GPU instances.
It uses the pre-trained, high-performance LightGBM model and Cross-Encoder cascade directly to produce valid submission files in **~5-8 minutes** without OOM risks.

### Kaggle Setup:
1. **Accelerator:** GPU T4 x2 (or GPU T4)
2. **Internet:** ON (for package installs and repo sync)
3. **Input Dataset:** Attached under  -> 


In [ ]:
# 1. Clone or Pull Latest Repository
import os
if not os.path.exists("/kaggle/working/amazon-entity-resolution"):
    !git clone https://github.com/Luciferxy/amazon-entity-resolution.git /kaggle/working/amazon-entity-resolution
else:
    %cd /kaggle/working/amazon-entity-resolution
    !git pull origin main

%cd /kaggle/working/amazon-entity-resolution/student_resource
print("Working directory:", os.getcwd())


In [ ]:
# 2. Check GPU & Hardware
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        dev = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"GPU {i}: {dev} | VRAM: {vram:.1f} GB")
else:
    print("WARNING: GPU accelerator not active. In Kaggle right panel, set Accelerator to GPU T4 x2.")


In [ ]:
# 3. Install Dependencies
%pip install -q pandas pyarrow numpy scipy scikit-learn lightgbm rapidfuzz anyascii sparse_dot_topn transformers
print("Dependencies installed successfully!")


In [ ]:
# 4. Verify Dataset Location
import sys
sys.path.insert(0, "src")
from config import DATASET
print(f"Detected Dataset Root: {DATASET}")
assert (DATASET / "test/test_source1.tsv").exists() or (DATASET / "train/train_source1.tsv").exists(), f"Dataset not found at {DATASET}! Click '+ Add Input' to attach the dataset."


In [ ]:
# 5. Configure Optimized Environment Variables for F0.5 >= 0.995
import os, torch
os.environ["ER_BACKEND"] = "gpu" if torch.cuda.is_available() else "cpu"
os.environ["ER_GPU_MODE"] = "svd"  # Fast ~3 min Tensor Core blocking
os.environ["ER_K"] = "7"  # K=7 for higher recall ceiling (>0.995 target)
os.environ["ER_FEAT_JOBS"] = "1"  # Safe single-process chunking to prevent OOM
os.environ["ER_PREP_WORKERS"] = "2"  # Safe preprocessing workers
os.environ["ER_CE_MODE"] = "cascade"  # Cross-Encoder rescores borderline pairs (0.40-0.85)
print("Configured GPU and Memory environment for F0.5 >= 0.995 target!")

In [ ]:
# 6. Run Test Inference & Generate Validated Outputs
!python src/run_test.py


In [ ]:
# 7. Copy Submission Files to /kaggle/working/ for 1-Click Download
!cp output/matching_results.tsv /kaggle/working/matching_results.tsv
!cp output/candidate_pairs.tsv /kaggle/working/candidate_pairs.tsv
!zip -j /kaggle/working/submission_files.zip output/matching_results.tsv output/candidate_pairs.tsv
print("
SUCCESS! Both files are saved to /kaggle/working/.")
print("Download 'matching_results.tsv' to upload to the Unstop leaderboard right away!")
!ls -lh /kaggle/working/*.tsv /kaggle/working/*.zip
